# Clinical Trial Footprint vs. Population Structure

**Course:** Data Science — Regular Exam, March 2026  
**Author:** Momchil Ivanov  
**Deadline:** 28 April 2026, 16:00h  
**Repository:** [github.com/Momchil-Ivanov/Data-Science-Mar-2026](https://github.com/Momchil-Ivanov/Data-Science-Mar-2026/tree/clinical-trial-footprint-vs-population-structure)

---

## Section 0 — Introduction & Problem Formulation

### 0.1 Motivation

Clinical trials are the gold standard for establishing the safety and efficacy of new medical treatments. Where these trials are conducted matters enormously: countries that host more trials gain earlier access to experimental therapies, attract research investment, and build local scientific capacity. Countries that are systematically underrepresented risk falling behind in medical innovation and having treatments developed without sufficient data from their populations.

This project asks a simple but revealing question: **is the global distribution of clinical trials proportional to where people actually live?** Or is it skewed — concentrated in wealthy, high-income countries regardless of population size?

We use two independent public data sources:

| Source | What it provides | Access |
|---|---|---|
| **ClinicalTrials.gov** | Trial recruitment site counts per country, phase, status, start year | REST API v2 — `https://clinicaltrials.gov/api/v2/` |
| **World Bank Indicators** | Population, GDP per capita, world region, income group | REST API v2 — `https://api.worldbank.org/v2/` |

Both sources are freely accessible without authentication and are maintained by authoritative international institutions (U.S. National Library of Medicine and the World Bank Group, respectively).

> **Reproducibility note:** No data files are committed to this repository. All data is fetched from the APIs above at runtime. Running Section 1 from top to bottom will recreate the full dataset.

---

### 0.2 Research Questions

We investigate four concrete questions. The appropriate statistical methods and formal hypotheses for each will be introduced in the relevant analysis section, after the data has been inspected.

1. **Q1 — Income groups:** How many trials per million inhabitants do different countries have, and how does this differ between income groups?
2. **Q2 — Population scaling:** Does a larger population proportionally lead to more trials, or is the relationship sub- or super-linear?
3. **Q3 — Regional inequality:** Is there a systematic difference in trial density across world regions, and how concentrated is the global distribution?
4. **Q4 — Temporal trends:** How has the picture changed over time (2000–2024), and is the gap between income groups growing or shrinking?

---

### 0.3 Prior Work

The unequal global distribution of clinical trials has been documented under the label of the **"10/90 gap"** — roughly 90 % of global health research funding addresses diseases affecting only 10 % of the world's population (Global Forum for Health Research, 2000). Viergever & Li (2015) mapped trial density globally using the WHO ICTRP registry and found a strong positive correlation with GDP per capita. Drain et al. (2014) showed that high-income countries account for the large majority of trial sites despite representing a minority of global disease burden.

This project extends that work by explicitly testing whether the population-to-trial relationship is proportional, and by applying distributional inequality tools (Lorenz curve, Gini coefficient) to quantify concentration rather than relying on regional averages alone.

In [8]:
import requests
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pycountry
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
import plotly.express as px

print('All imports OK')

All imports OK


## Section 1 — Data Ingestion

### 1.1 ClinicalTrials.gov API v2

We fetch trial metadata directly from the ClinicalTrials.gov REST API v2. No files are stored in the repository — data is recreated at runtime on every run.

**What we extract per study:**
- `nctId` — unique trial identifier
- `startDate` — trial start date (we keep the year)
- `locations[].country` — every country where the trial recruits participants

**Unit of measurement:** a single multinational trial with sites in 30 countries contributes **1 count to each of those 30 countries**. We therefore measure *trial-site presence per country*, not globally unique trials. This is a deliberate choice: it reflects the research activity actually experienced by a given country's population.

**Filters applied at fetch time:**
- `overallStatus`: `COMPLETED`, `ACTIVE_NOT_RECRUITING`, `RECRUITING` — excludes withdrawn/terminated studies with no recruitment activity.

In [9]:
CT_BASE = "https://clinicaltrials.gov/api/v2/studies"

CT_PARAMS = {
    "format": "json",
    "pageSize": 1000,
    "fields": "NCTId,StartDate,LocationCountry",
    "filter.overallStatus": "COMPLETED,ACTIVE_NOT_RECRUITING,RECRUITING",
}


def ct_get_page(params: dict, max_retries: int = 5) -> dict:
    """Fetch one ClinicalTrials.gov page, retrying transient network failures."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(CT_BASE, params=params, timeout=(15, 90))
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as exc:
            if attempt == max_retries:
                page_token = params.get("pageToken", "first page")
                raise RuntimeError(f"ClinicalTrials.gov request failed at {page_token}") from exc

            wait_seconds = 2 ** attempt
            print(f"ClinicalTrials.gov request failed; retrying in {wait_seconds}s ({attempt}/{max_retries})")
            time.sleep(wait_seconds)


def fetch_clinicaltrials() -> pd.DataFrame:
    """Page through ClinicalTrials.gov API v2 and return one row per
    (nct_id, country, start_year) triplet."""
    records = []
    params = CT_PARAMS.copy()
    page_count = 0

    while True:
        data = ct_get_page(params)
        page_count += 1

        for study in data.get("studies", []):
            proto = study.get("protocolSection", {})
            nct_id = proto.get("identificationModule", {}).get("nctId", "")
            start_raw = proto.get("statusModule", {}).get("startDateStruct", {}).get("date", "")
            start_year = int(start_raw[:4]) if start_raw and len(start_raw) >= 4 else None
            locations = proto.get("contactsLocationsModule", {}).get("locations", [])
            countries = {loc.get("country") for loc in locations if loc.get("country")}
            for country in countries:
                records.append({"nct_id": nct_id, "country": country, "start_year": start_year})

        if page_count % 25 == 0:
            print(f"Fetched {page_count:,} pages so far...")

        next_token = data.get("nextPageToken")
        if not next_token:
            break
        params["pageToken"] = next_token
        time.sleep(0.2)  # stay within API rate limits

    df = pd.DataFrame(records)
    print(f"Fetched {df['nct_id'].nunique():,} unique studies → {len(df):,} (study, country) pairs")
    print(f"Unique country names in raw data: {df['country'].nunique()}")
    return df

ct_raw = fetch_clinicaltrials()
ct_raw.head()

Fetched 25 pages so far...
Fetched 50 pages so far...
Fetched 75 pages so far...
Fetched 100 pages so far...
Fetched 125 pages so far...
Fetched 150 pages so far...
Fetched 175 pages so far...
Fetched 200 pages so far...
Fetched 225 pages so far...
Fetched 250 pages so far...
Fetched 275 pages so far...
Fetched 300 pages so far...
Fetched 325 pages so far...
Fetched 350 pages so far...
Fetched 375 pages so far...
Fetched 400 pages so far...
Fetched 380,990 unique studies → 576,982 (study, country) pairs
Unique country names in raw data: 219


,nct_id,country,start_year
0,NCT00478192,United States,2007.0
1,NCT00478192,India,2007.0
2,NCT05189171,United States,2022.0
3,NCT05077436,United Kingdom,2021.0
4,NCT06766136,Turkey (Türkiye),2024.0


### 1.2 World Bank Indicators API

We fetch country-level demographic and economic context directly from the World Bank REST API. These variables are needed to compare clinical trial activity against the size and development level of each country's population.

**What we extract per country:**
- `iso3` — ISO 3166-1 alpha-3 country code, used as the stable merge key
- `country` — World Bank country name
- `region` — World Bank region classification
- `income_group` — World Bank income group classification
- `population` — total population (`SP.POP.TOTL`)
- `gdp_per_capita_usd` — GDP per capita in current US dollars (`NY.GDP.PCAP.CD`)

We use **2023** as the reference year because it is recent while usually having broader coverage than the most recent calendar year. Aggregates such as "World" or "High income" are excluded, leaving only country-level records.

In [10]:
WB_BASE = "https://api.worldbank.org/v2"
WB_YEAR = 2023

WB_INDICATORS = {
    "population": "SP.POP.TOTL",
    "gdp_per_capita_usd": "NY.GDP.PCAP.CD",
}


def wb_get(path: str, params: dict | None = None) -> list[dict]:
    """Request one World Bank API endpoint and return the data payload."""
    request_params = {"format": "json", "per_page": 1000}
    if params:
        request_params.update(params)

    resp = requests.get(f"{WB_BASE}/{path}", params=request_params, timeout=30)
    resp.raise_for_status()
    payload = resp.json()

    if not isinstance(payload, list) or len(payload) < 2:
        return []
    return payload[1] or []


def fetch_country_metadata() -> pd.DataFrame:
    """Fetch country names, ISO codes, regions, and income groups."""
    records = []

    for item in wb_get("country"):
        region = item.get("region", {})
        income_group = item.get("incomeLevel", {})

        # World Bank aggregate regions have region id "NA"; keep countries only.
        if region.get("id") == "NA":
            continue

        records.append(
            {
                "iso3": item.get("id"),
                "iso2": item.get("iso2Code"),
                "country": item.get("name"),
                "region": region.get("value"),
                "income_group": income_group.get("value"),
            }
        )

    return pd.DataFrame(records)


def fetch_wb_indicator(indicator: str, value_col: str, year: int = WB_YEAR) -> pd.DataFrame:
    """Fetch one World Bank indicator for all countries in a given year."""
    records = []

    for item in wb_get(f"country/all/indicator/{indicator}", {"date": str(year)}):
        iso3 = item.get("countryiso3code")
        value = item.get("value")
        if iso3 and value is not None:
            records.append({"iso3": iso3, value_col: value})

    return pd.DataFrame(records)


wb_meta = fetch_country_metadata()
wb_pop = fetch_wb_indicator(WB_INDICATORS["population"], "population")
wb_gdp = fetch_wb_indicator(WB_INDICATORS["gdp_per_capita_usd"], "gdp_per_capita_usd")

wb_raw = wb_meta.merge(wb_pop, on="iso3", how="left").merge(wb_gdp, on="iso3", how="left")

print(f"Fetched World Bank metadata for {len(wb_raw):,} countries")
print(f"Countries with population in {WB_YEAR}: {wb_raw['population'].notna().sum():,}")
print(f"Countries with GDP per capita in {WB_YEAR}: {wb_raw['gdp_per_capita_usd'].notna().sum():,}")

wb_raw.head()

Fetched World Bank metadata for 217 countries
Countries with population in 2023: 217
Countries with GDP per capita in 2023: 203


,iso3,iso2,country,region,income_group,population,gdp_per_capita_usd
0,ABW,AW,Aruba,Latin America & Caribbean,High income,107359,35718.753119
1,AFG,AF,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,41454761,413.757895
2,AGO,AO,Angola,Sub-Saharan Africa,Lower middle income,36749906,2916.136633
3,ALB,AL,Albania,Europe & Central Asia,Upper middle income,2414095,9730.869219
4,AND,AD,Andorra,Europe & Central Asia,High income,80856,46812.426101


## Section 2 — Data Inspection & Cleaning

### 2.1 Inspect raw ClinicalTrials.gov data

Before cleaning or merging, we inspect the raw ClinicalTrials.gov extraction. This tells us:

- how many rows and columns were fetched;
- which data types pandas assigned;
- whether key fields are missing;
- what year range the trials cover;
- which country names appear most often.

This step is important because the next cleaning step depends on what the raw API data actually looks like.

In [11]:
print("=== ClinicalTrials.gov raw data shape ===")
print(ct_raw.shape)

print("\n=== Column data types ===")
print(ct_raw.dtypes)

print("\n=== Missing values ===")
print(ct_raw.isnull().sum())

print("\n=== Start year coverage ===")
print(ct_raw["start_year"].describe())

print("\n=== Top 20 raw country names ===")
print(ct_raw["country"].value_counts().head(20))

=== ClinicalTrials.gov raw data shape ===
(576982, 3)

=== Column data types ===
nct_id         object
country        object
start_year    float64
dtype: object

=== Missing values ===
nct_id           0
country          0
start_year    2367
dtype: int64

=== Start year coverage ===
count    574615.000000
mean       2016.075122
std           6.760820
min        1916.000000
25%        2011.000000
50%        2017.000000
75%        2022.000000
max        2030.000000
Name: start_year, dtype: float64

=== Top 20 raw country names ===
country
United States       150093
France               30677
China                28812
Canada               24551
Germany              22277
United Kingdom       21713
Turkey (Türkiye)     19321
Spain                18995
Italy                18862
South Korea          12818
Belgium              11820
Netherlands          11478
Egypt                10710
Australia             9795
Denmark               9774
Poland                8989
Taiwan                840

### 2.2 Standardise ClinicalTrials.gov country names

ClinicalTrials.gov gives country names as free-text strings, while the World Bank data uses standard ISO country codes. To merge both sources later, we need a shared country key.

We convert each ClinicalTrials.gov country name to an ISO2 code using `pycountry`. Some country names require manual overrides because API naming conventions differ from ISO naming conventions (for example, `Turkey (Türkiye)`, `South Korea`, or `Taiwan`).

The result of this step is a new `iso2` column in `ct_raw`. Any unresolved country names are printed so they can be inspected before the merge.

In [12]:
CT_COUNTRY_OVERRIDES = {
    "Bolivia": "BO",
    "Congo, The Democratic Republic of the": "CD",
    "Czechia": "CZ",
    "Hong Kong": "HK",
    "Iran": "IR",
    "Kosovo": "XK",
    "Moldova": "MD",
    "North Korea": "KP",
    "Palestine": "PS",
    "Russia": "RU",
    "South Korea": "KR",
    "Taiwan": "TW",
    "Tanzania": "TZ",
    "Turkey (Türkiye)": "TR",
    "Venezuela": "VE",
    "Vietnam": "VN",
}


def country_to_iso2(country_name: str) -> str | None:
    """Map a ClinicalTrials.gov country name to an ISO2 country code."""
    if pd.isna(country_name):
        return None

    country_name = str(country_name).strip()

    if country_name in CT_COUNTRY_OVERRIDES:
        return CT_COUNTRY_OVERRIDES[country_name]

    try:
        return pycountry.countries.lookup(country_name).alpha_2
    except LookupError:
        return None


ct_raw["iso2"] = ct_raw["country"].map(country_to_iso2)

unresolved = (
    ct_raw.loc[ct_raw["iso2"].isna(), "country"]
    .value_counts()
    .rename_axis("country")
    .reset_index(name="rows")
)

print(f"Mapped ISO2 codes for {ct_raw['iso2'].notna().sum():,} rows")
print(f"Unresolved rows: {ct_raw['iso2'].isna().sum():,}")
print(f"Unresolved unique country names: {len(unresolved):,}")

unresolved.head(20)

Mapped ISO2 codes for 576,389 rows
Unresolved rows: 593
Unresolved unique country names: 17


,country,rows
0,Democratic Republic of the Congo,141
1,Reunion,89
2,Côte d’Ivoire,86
3,The Gambia,79
4,Palestinian Territories,71
5,Burma,36
6,Serbia and Montenegro,34
7,The Bahamas,16
8,Macau,15
9,Brunei,8


### 2.3 Build cleaned ClinicalTrials.gov trial counts

Now that each ClinicalTrials.gov row has a standard country code, we build the cleaned trial-count tables used later in the analysis.

Cleaning rules:

- remove rows without an `iso2` country code;
- remove rows without `start_year`;
- keep trials from **2000–2024** only;
- convert `start_year` from float to integer;
- aggregate to two useful tables:
  - `ct_panel`: trial counts by country and year;
  - `ct_total`: total trial counts by country across the whole analysis period.

The year filter is needed because the raw data includes very old records and future planned trials; the project focuses on the modern ClinicalTrials.gov era.

In [ ]:
ANALYSIS_START_YEAR = 2000
ANALYSIS_END_YEAR = 2024

ct_clean = (
    ct_raw
    .dropna(subset=["iso2", "start_year"])
    .query("@ANALYSIS_START_YEAR <= start_year <= @ANALYSIS_END_YEAR")
    .copy()
)

ct_clean["start_year"] = ct_clean["start_year"].astype(int)

ct_panel = (
    ct_clean
    .groupby(["iso2", "start_year"], as_index=False)
    .agg(trial_count=("nct_id", "count"))
)

ct_total = (
    ct_clean
    .groupby("iso2", as_index=False)
    .agg(trial_count=("nct_id", "count"))
)

print(f"Rows before cleaning: {len(ct_raw):,}")
print(f"Rows after cleaning: {len(ct_clean):,}")
print(f"Dropped rows: {len(ct_raw) - len(ct_clean):,}")
print(f"Country-year rows in ct_panel: {len(ct_panel):,}")
print(f"Countries in ct_total: {len(ct_total):,}")

ct_total.sort_values("trial_count", ascending=False).head(10)